In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import MinMaxScaler
from skopt import BayesSearchCV
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.metrics import r2_score

# Load dataset
df = pd.read_csv("tesla.csv")
df.head()

# Define features and target
x = df[['Open', 'High', 'Low']]
y = df[['Close']]

# Normalize features and target
scaler_x = MinMaxScaler()
x_encode = scaler_x.fit_transform(x[['Open', 'High', 'Low']])

scaler_y = MinMaxScaler()
y = scaler_y.fit_transform(y)

# Split dataset
x_train, x_test, y_train, y_test = train_test_split(x_encode, y, 
                                                    test_size=0.3, 
                                                    random_state=42)

# Initialize model
model = DecisionTreeRegressor()

# Define hyperparameter grid
param_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# Grid Search
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(x_train, y_train)

# Randomized Search
random_search = RandomizedSearchCV(model, param_distributions=param_grid, 
                                   n_iter=10, 
                                   cv=5, 
                                   scoring='r2', 
                                   n_jobs=-1, 
                                   random_state=42)
random_search.fit(x_train, y_train)

# Bayesian Optimization
bayes_search = BayesSearchCV(model, param_grid, cv=5, 
                             scoring='r2', 
                             n_iter=10, 
                             n_jobs=-1, 
                             random_state=42)
bayes_search.fit(x_train, y_train)

# Hyperband
hyperband = HalvingGridSearchCV(model, param_grid, cv=5, 
                                factor=2, 
                                scoring='r2', 
                                n_jobs=-1)
hyperband.fit(x_train, y_train)

# Print best parameters
print("Grid Search Best Parameters:", grid_search.best_params_)
print("Random Search Best Parameters:", random_search.best_params_)
print("Bayesian Optimization Best Parameters:", bayes_search.best_params_)
print("Hyperband Best Parameters:", hyperband.best_params_)

Grid Search Best Parameters: {'max_depth': 10, 'max_features': None, 'min_samples_leaf': 2, 'min_samples_split': 2}
Random Search Best Parameters: {'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 10}
Bayesian Optimization Best Parameters: OrderedDict({'max_depth': None, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 5})
Hyperband Best Parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2}


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


In [4]:
# Train best model
best_model = DecisionTreeRegressor(**grid_search.best_params_)
best_model.fit(x_train, y_train)

# Make predictions
y_pred = best_model.predict(x_test)

# Evaluate model
r2 = r2_score(y_test, y_pred)
print(f"R² Score: {r2:.4f}")

R² Score: 0.9992
